In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver  
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:llama-3.3-70b-versatile",
            trigger= ("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
config = {"configurable":{"thread_id":"test_1"}}

In [4]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='bca93263-8530-434d-805c-0315b27bfb4c'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.011571517, 'completion_tokens_details': None, 'prompt_time': 0.001967986, 'prompt_tokens_details': None, 'queue_time': 0.161728324, 'total_time': 0.013539503}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd6a8-584f-7252-a3e0-71286789e1e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='bca93263-8530-434d-805c-0315b27bfb4c'), AIMess

## Human in the loop middleware

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id:str) -> str:
    """Mock function to real an email by it's email id"""
    return f"Email content for id :{email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock Function to send an email"""
    return f"Enail sent to {recipient} with the subject '{subject}'"



In [6]:
from pydantic_core.core_schema import MultiHostUrlSchema
agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    tools = [read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
            "send_email_tool":{
                "allowed_decision": ["approved","editied","rejected"]
            },
            "read_email_tool": False
            
            }
        )
    ]
)

In [12]:
config = {"configurable":{"thread_id":"test-approve"}}

result = agent.invoke(
    {"messages":[HumanMessage(content="Send email to goyalmayank492@gmail.com with subject 'Hello' and body 'How are you' ")]},
    config=config
)

In [13]:
result

{'messages': [HumanMessage(content="Sned email to john@gmail.com with subject 'Hello' and body 'How are you' ", additional_kwargs={}, response_metadata={}, id='eb499a9f-f485-471c-96f9-b5496bbf874c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1x8hzs4n', 'function': {'arguments': '{"body":"How are you","recipient":"john@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 315, 'total_tokens': 347, 'completion_time': 0.069368362, 'completion_tokens_details': None, 'prompt_time': 0.01552674, 'prompt_tokens_details': None, 'queue_time': 0.05023793, 'total_time': 0.084895102}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd6c3-9dbd-7b52-b0dd-b207f71a51a8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo

In [14]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

In [15]:
result

{'messages': [HumanMessage(content="Sned email to john@gmail.com with subject 'Hello' and body 'How are you' ", additional_kwargs={}, response_metadata={}, id='eb499a9f-f485-471c-96f9-b5496bbf874c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q1x8hzs4n', 'function': {'arguments': '{"body":"How are you","recipient":"john@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 315, 'total_tokens': 347, 'completion_time': 0.069368362, 'completion_tokens_details': None, 'prompt_time': 0.01552674, 'prompt_tokens_details': None, 'queue_time': 0.05023793, 'total_time': 0.084895102}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd6c3-9dbd-7b52-b0dd-b207f71a51a8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'bo